# Notebook 01 — Temporal data audit

Audit temporal integrity and structural data quality for the assembled checkpoint produced by Notebook 00.


## 0. Imports and setup


In [ ]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


In [ ]:
# Dependency preflight: parquet support (read/write)
try:
    import pyarrow  # noqa: F401
except ImportError as exc:
    raise ImportError(
        "Missing optional dependency 'pyarrow' required for parquet IO in this notebook. "
        "Install with: pip install pyarrow"
    ) from exc


In [ ]:
def find_project_root(start_path: Path) -> Path:
    """Find the project root by walking upward from a start path.

    Markers searched (in priority order):
    - pyproject.toml
    - README.md
    - data/ directory

    The notebook must not assume the execution CWD is already the project root.
    """

    best_candidate = None
    best_score = -1

    for candidate in [start_path, *start_path.parents]:
        has_pyproject = (candidate / 'pyproject.toml').exists()
        has_readme = (candidate / 'README.md').exists()
        has_data_dir = (candidate / 'data').is_dir()

        score = 0
        score += 4 if has_pyproject else 0
        score += 2 if has_readme else 0
        score += 1 if has_data_dir else 0

        if score > best_score:
            best_candidate = candidate
            best_score = score

        if has_data_dir and (has_pyproject or has_readme):
            return candidate

    if best_candidate is not None and (best_candidate / 'data').is_dir():
        return best_candidate

    raise FileNotFoundError(
        "Could not locate PROJECT_ROOT by walking upward from the current working directory. "
        "Expected to find at least a 'data/' directory and ideally 'README.md' or 'pyproject.toml'."
    )


In [ ]:
PROJECT_ROOT = find_project_root(Path.cwd().resolve())

ASSEMBLED_PATH = PROJECT_ROOT / 'data' / 'interim' / 'beijing_air_quality' / 'beijing_multisite_assembled.parquet'
MANIFEST_PATH = PROJECT_ROOT / 'data' / 'interim' / 'beijing_air_quality' / 'assembly_manifest.json'

AUDIT_ROOT = PROJECT_ROOT / 'data' / 'interim' / 'beijing_air_quality' / 'temporal_audit'

# Station policy thresholds (structural audit only; not modeling verdicts)
TH_INCLUDE_MAX_MISSING_HOUR_RATE = 0.01
TH_INCLUDE_MAX_PM25_MISSING_RATE = 0.10
TH_EXCLUDE_MISSING_HOUR_RATE = 0.10
TH_EXCLUDE_PM25_MISSING_RATE = 0.50

# Sanity severity thresholds (rate of flagged values)
TH_SEVERE_NEGATIVE_RATE = 0.01
TH_SEVERE_PLAUSIBILITY_RATE = 0.01

# Broad plausibility bounds (screening only; not corrections)
TEMP_BOUNDS_C = (-50.0, 50.0)
PRES_BOUNDS_HPA = (800.0, 1100.0)

TOP_GAPS_PER_STATION = 5

REQUIRED_COLUMNS = ['station', 'timestamp', 'PM2.5']

KEY_COLUMNS = ['station', 'timestamp']
POLLUTANTS = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3']
METEOROLOGY = ['TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM', 'wd']

CANONICAL_COLUMNS_NOTEBOOK_00 = [
    'station', 'timestamp', 'year', 'month', 'day', 'hour', 'No', 'PM2.5',
    'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN',
    'wd', 'WSPM', 'source_file',
]

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'ASSEMBLED_PATH: {ASSEMBLED_PATH}')
print(f'MANIFEST_PATH: {MANIFEST_PATH}')


## 1. Notebook scope and audit contract

This notebook audits the assembled checkpoint produced by **Notebook 00**.

Audit-only scope:
- temporal integrity (hourly grid alignment, continuity, missing station-hours)
- structural data quality checks (keys, duplicates, missingness, basic value sanity flags)
- provisional **station status** assignment: `include` / `caution` / `exclude_candidate`

This notebook does **not** perform EDA storytelling, imputation, split design, target construction, lag/feature engineering, baselines, leakage analysis, or modeling.

Missingness and value anomalies are surfaced here but are not fixed here.

Notebook 02 will handle environmental EDA using the audit outputs.

Station statuses produced here are **provisional structural audit statuses**, not modeling verdicts.

AI assistance disclosure: notebook structure and code were AI-assisted; correctness should be verified by executing the notebook end-to-end and inspecting the persisted artifacts and blocker/caution outputs.


## 2. Load assembled dataset and assembly manifest


In [ ]:
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f'Missing assembly manifest: {MANIFEST_PATH}')

with MANIFEST_PATH.open('r', encoding='utf-8') as f:
    assembly_manifest = json.load(f)

if not ASSEMBLED_PATH.exists():
    raise FileNotFoundError(f'Missing assembled parquet: {ASSEMBLED_PATH}')

df = pd.read_parquet(ASSEMBLED_PATH)

manifest_points_to = (assembly_manifest.get('artifact_paths', {}) or {}).get('assembled_parquet')
expected_relative = ASSEMBLED_PATH.relative_to(PROJECT_ROOT).as_posix()
manifest_parquet_path_match = (manifest_points_to == expected_relative)

if not manifest_parquet_path_match:
    print('Manifest parquet path mismatch:')
    print(f'- manifest: {manifest_points_to}')
    print(f'- expected: {expected_relative}')

df.shape


## 3. Core schema, key, and timestamp contract checks


In [ ]:
missing_required_columns = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing_required_columns:
    raise ValueError(f'Missing required columns: {missing_required_columns}')

missing_canonical_columns = [c for c in CANONICAL_COLUMNS_NOTEBOOK_00 if c not in df.columns]
if missing_canonical_columns:
    print('Canonical columns from Notebook 00 missing (caution):')
    print(missing_canonical_columns)

if not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
    parsed = pd.to_datetime(df['timestamp'], errors='coerce')
    if int(parsed.isna().sum()) > 0:
        raise ValueError('timestamp could not be parsed cleanly (nulls introduced)')
    df = df.copy()
    df['timestamp'] = parsed

key_null_counts = {
    'station': int(df['station'].isna().sum()),
    'timestamp': int(df['timestamp'].isna().sum()),
}
if key_null_counts['station'] > 0 or key_null_counts['timestamp'] > 0:
    raise ValueError(f'Null key values found: {key_null_counts}')

station_as_str = df['station'].astype(str)
station_stripped = station_as_str.str.strip()
whitespace_station_rows = int((station_as_str != station_stripped).sum())
if whitespace_station_rows > 0:
    raise ValueError(f'Station tokens contain leading/trailing whitespace for {whitespace_station_rows} rows')

{
    'n_rows': int(len(df)),
    'n_stations': int(df['station'].nunique()),
    'manifest_parquet_path_match': bool(manifest_parquet_path_match),
}


## 4. Station inventory and coverage window


In [ ]:
station_counts = df.groupby('station').size().rename('row_count').reset_index()
station_bounds = df.groupby('station')['timestamp'].agg(ts_min='min', ts_max='max').reset_index()
station_inventory = station_counts.merge(station_bounds, on='station', how='inner')

if station_inventory.empty:
    raise ValueError('No valid stations found after basic contract checks')

station_inventory.sort_values('station')


## 5. Hourly-grid alignment and timestamp continuity


In [ ]:
ts = df['timestamp']
non_hourly_component_rows = int(((ts.dt.minute != 0) | (ts.dt.second != 0) | (ts.dt.microsecond != 0)).sum())
if non_hourly_component_rows > 0:
    raise ValueError(f'Found {non_hourly_component_rows} timestamps with non-zero minute/second/microsecond')

off_grid_rows = int((ts != ts.dt.floor('h')).sum())
if off_grid_rows > 0:
    raise ValueError(f'Found {off_grid_rows} timestamps not aligned to hourly grid')

{'non_hourly_component_rows': non_hourly_component_rows, 'off_grid_rows': off_grid_rows}


## 6. Missing station-hour audit


In [ ]:
df_sorted = df.sort_values(['station', 'timestamp']).reset_index(drop=True)

missing_station_hours_rows = []
coverage_rows = []

for station_name, group in df_sorted.groupby('station', sort=True):
    ts_min = group['timestamp'].min()
    ts_max = group['timestamp'].max()

    expected = pd.date_range(ts_min, ts_max, freq='h')
    observed = pd.Index(group['timestamp'].unique()).sort_values()

    expected_hours = int(len(expected))
    observed_hours = int(len(observed))

    missing = expected.difference(observed)
    missing_hours = int(len(missing))
    missing_rate = float(missing_hours / expected_hours) if expected_hours else float('nan')

    coverage_rows.append({
        'station': station_name,
        'row_count': int(len(group)),
        'ts_min': ts_min,
        'ts_max': ts_max,
        'expected_hours': expected_hours,
        'observed_hours': observed_hours,
        'missing_hours': missing_hours,
        'missing_rate': missing_rate,
    })

    if missing_hours > 0:
        for ts_missing in missing:
            missing_station_hours_rows.append({'station': station_name, 'timestamp': ts_missing})

station_coverage_summary = pd.DataFrame(coverage_rows).sort_values('station').reset_index(drop=True)
missing_station_hours = pd.DataFrame(missing_station_hours_rows)

station_coverage_summary.head()


In [ ]:
def compute_gap_segments_top(missing_df: pd.DataFrame, top_n: int) -> pd.DataFrame:
    if missing_df.empty:
        return pd.DataFrame(
            columns=['station', 'gap_start', 'gap_end', 'gap_length_hours', 'rank_within_station']
        )

    segments = []

    for station_name, group in missing_df.groupby('station', sort=True):
        ts_sorted = pd.Series(pd.to_datetime(group['timestamp']).sort_values().unique())
        if ts_sorted.empty:
            continue

        start = ts_sorted.iloc[0]
        prev = ts_sorted.iloc[0]

        for current in ts_sorted.iloc[1:]:
            if (current - prev) == pd.Timedelta(hours=1):
                prev = current
                continue

            gap_start = start
            gap_end = prev
            gap_len = int(((gap_end - gap_start) / pd.Timedelta(hours=1)) + 1)
            segments.append({
                'station': station_name,
                'gap_start': gap_start,
                'gap_end': gap_end,
                'gap_length_hours': gap_len,
            })

            start = current
            prev = current

        gap_start = start
        gap_end = prev
        gap_len = int(((gap_end - gap_start) / pd.Timedelta(hours=1)) + 1)
        segments.append({
            'station': station_name,
            'gap_start': gap_start,
            'gap_end': gap_end,
            'gap_length_hours': gap_len,
        })

    seg_df = pd.DataFrame(segments)
    if seg_df.empty:
        return seg_df

    seg_df = seg_df.sort_values(['station', 'gap_length_hours'], ascending=[True, False])
    seg_df['rank_within_station'] = seg_df.groupby('station').cumcount() + 1
    seg_df = seg_df[seg_df['rank_within_station'] <= top_n].reset_index(drop=True)
    return seg_df


In [ ]:
gap_segments_top = compute_gap_segments_top(missing_station_hours, TOP_GAPS_PER_STATION)
gap_segments_top.head()


## 7. Missing-value audit by variable family


In [ ]:
def variable_family_for_column(column_name: str) -> str:
    if column_name in KEY_COLUMNS:
        return 'key'
    if column_name in POLLUTANTS:
        return 'pollutant'
    if column_name in METEOROLOGY:
        return 'meteorology'
    return 'other'


In [ ]:
missingness_overall_rows = []
for col in df.columns:
    missing_count = int(df[col].isna().sum())
    missing_rate = float(missing_count / len(df)) if len(df) else float('nan')
    missingness_overall_rows.append({
        'column': col,
        'missing_count': missing_count,
        'missing_rate': missing_rate,
        'variable_family': variable_family_for_column(col),
    })

missingness_overall = pd.DataFrame(missingness_overall_rows).sort_values(
    ['variable_family', 'missing_rate', 'column'], ascending=[True, False, True]
)

missingness_by_station_rows = []
for station_name, group in df.groupby('station', sort=True):
    n = int(len(group))
    for col in df.columns:
        missing_count = int(group[col].isna().sum())
        missing_rate = float(missing_count / n) if n else float('nan')
        missingness_by_station_rows.append({
            'station': station_name,
            'column': col,
            'missing_count': missing_count,
            'missing_rate': missing_rate,
            'variable_family': variable_family_for_column(col),
        })

missingness_by_station = pd.DataFrame(missingness_by_station_rows)

missingness_overall.head(10)


## 8. Duplicate-key and ordering checks


In [ ]:
duplicate_groups = (
    df.groupby(['station', 'timestamp'], dropna=False)
    .size()
    .reset_index(name='row_count')
)
duplicate_groups = duplicate_groups[duplicate_groups['row_count'] > 1].copy()

duplicate_key_count_overall = int(duplicate_groups['row_count'].sum())
duplicate_key_count_by_station = (
    duplicate_groups.groupby('station')['row_count'].sum().astype(int).to_dict()
    if not duplicate_groups.empty
    else {}
)

if duplicate_key_count_overall > 0:
    raise ValueError(
        f'Duplicate (station, timestamp) keys exist (overall duplicate rows: {duplicate_key_count_overall})'
    )

df_sorted = df.sort_values(['station', 'timestamp']).reset_index(drop=True)
non_monotonic_stations = []
for station_name, group in df_sorted.groupby('station', sort=True):
    if not group['timestamp'].is_monotonic_increasing:
        non_monotonic_stations.append(station_name)

if non_monotonic_stations:
    raise ValueError(
        f'Non-monotonic timestamps found after sorting for stations: {non_monotonic_stations}'
    )

{
    'duplicate_key_count_overall': duplicate_key_count_overall,
    'non_monotonic_station_count': len(non_monotonic_stations),
}


## 9. Structural value sanity checks


In [ ]:
def rate(count: int, denom: int) -> float:
    return float(count / denom) if denom else float('nan')


In [ ]:
sanity_rows = []
n_total = int(len(df))

negative_columns = list(dict.fromkeys(POLLUTANTS + ['RAIN', 'WSPM']))
for col in negative_columns:
    if col not in df.columns:
        continue

    overall_flags = int((df[col] < 0).sum(skipna=True))
    sanity_rows.append({
        'check_name': 'negative_values',
        'column': col,
        'station': None,
        'flag_count': overall_flags,
        'flag_rate': rate(overall_flags, n_total),
        'severity': 'severe' if rate(overall_flags, n_total) >= TH_SEVERE_NEGATIVE_RATE else 'non_severe',
    })

    for station_name, group in df.groupby('station', sort=True):
        n = int(len(group))
        flags = int((group[col] < 0).sum(skipna=True))
        sanity_rows.append({
            'check_name': 'negative_values',
            'column': col,
            'station': station_name,
            'flag_count': flags,
            'flag_rate': rate(flags, n),
            'severity': 'severe' if rate(flags, n) >= TH_SEVERE_NEGATIVE_RATE else 'non_severe',
        })

if 'TEMP' in df.columns:
    lo, hi = TEMP_BOUNDS_C
    overall_flags = int(((df['TEMP'] < lo) | (df['TEMP'] > hi)).sum(skipna=True))
    sanity_rows.append({
        'check_name': 'temp_out_of_bounds',
        'column': 'TEMP',
        'station': None,
        'flag_count': overall_flags,
        'flag_rate': rate(overall_flags, n_total),
        'severity': 'severe' if rate(overall_flags, n_total) >= TH_SEVERE_PLAUSIBILITY_RATE else 'non_severe',
    })
    for station_name, group in df.groupby('station', sort=True):
        n = int(len(group))
        flags = int(((group['TEMP'] < lo) | (group['TEMP'] > hi)).sum(skipna=True))
        sanity_rows.append({
            'check_name': 'temp_out_of_bounds',
            'column': 'TEMP',
            'station': station_name,
            'flag_count': flags,
            'flag_rate': rate(flags, n),
            'severity': 'severe' if rate(flags, n) >= TH_SEVERE_PLAUSIBILITY_RATE else 'non_severe',
        })

if 'PRES' in df.columns:
    lo, hi = PRES_BOUNDS_HPA
    overall_flags = int(((df['PRES'] < lo) | (df['PRES'] > hi)).sum(skipna=True))
    sanity_rows.append({
        'check_name': 'pres_out_of_bounds',
        'column': 'PRES',
        'station': None,
        'flag_count': overall_flags,
        'flag_rate': rate(overall_flags, n_total),
        'severity': 'severe' if rate(overall_flags, n_total) >= TH_SEVERE_PLAUSIBILITY_RATE else 'non_severe',
    })
    for station_name, group in df.groupby('station', sort=True):
        n = int(len(group))
        flags = int(((group['PRES'] < lo) | (group['PRES'] > hi)).sum(skipna=True))
        sanity_rows.append({
            'check_name': 'pres_out_of_bounds',
            'column': 'PRES',
            'station': station_name,
            'flag_count': flags,
            'flag_rate': rate(flags, n),
            'severity': 'severe' if rate(flags, n) >= TH_SEVERE_PLAUSIBILITY_RATE else 'non_severe',
        })

if 'wd' in df.columns:
    wd_as_str = df['wd'].astype(str)
    wd_stripped = wd_as_str.str.strip()
    empty_like = (df['wd'].isna()) | (wd_stripped == '')

    overall_flags = int(empty_like.sum())
    sanity_rows.append({
        'check_name': 'wd_null_or_empty',
        'column': 'wd',
        'station': None,
        'flag_count': overall_flags,
        'flag_rate': rate(overall_flags, n_total),
        'severity': 'non_severe',
    })

    for station_name, group in df.groupby('station', sort=True):
        n = int(len(group))
        wd_as_str_g = group['wd'].astype(str)
        wd_stripped_g = wd_as_str_g.str.strip()
        empty_like_g = (group['wd'].isna()) | (wd_stripped_g == '')
        flags = int(empty_like_g.sum())
        sanity_rows.append({
            'check_name': 'wd_null_or_empty',
            'column': 'wd',
            'station': station_name,
            'flag_count': flags,
            'flag_rate': rate(flags, n),
            'severity': 'non_severe',
        })

value_sanity_flags = pd.DataFrame(sanity_rows)
value_sanity_flags.head(10)


In [ ]:
# Optional wind-direction token inventory (no recoding)
if 'wd' in df.columns:
    import re

    wd_as_str = df['wd'].astype(str)
    wd_stripped = wd_as_str.str.strip()
    empty_like = (df['wd'].isna()) | (wd_stripped == '')

    wd_unique_tokens = sorted(set(wd_stripped[~empty_like].unique().tolist()))
    # Structural 'unexpected' = does not match 1-3 uppercase letters (e.g., N, NNW, ESE)
    wd_unexpected_tokens = [t for t in wd_unique_tokens if re.match(r'^[A-Z]{1,3}$', t) is None]

    wd_token_note = {
        'wd_unique_token_count': int(len(wd_unique_tokens)),
        'wd_unexpected_token_count': int(len(wd_unexpected_tokens)),
        'wd_unexpected_tokens': wd_unexpected_tokens,
    }

    # Append a compact summary row into value_sanity_flags (station=None).
    extra = pd.DataFrame([
        {
            'check_name': 'wd_unexpected_tokens',
            'column': 'wd',
            'station': None,
            'flag_count': wd_token_note['wd_unexpected_token_count'],
            'flag_rate': float('nan'),
            'severity': 'non_severe',
        }
    ])
    value_sanity_flags = pd.concat([value_sanity_flags, extra], ignore_index=True)

    wd_token_note
else:
    {'wd_unique_token_count': 0, 'wd_unexpected_token_count': 0, 'wd_unexpected_tokens': []}


## 10. Station status policy


In [ ]:
pm25_missing_by_station = (
    missingness_by_station[missingness_by_station['column'] == 'PM2.5']
    .set_index('station')['missing_rate']
    .to_dict()
)

severe_sanity_by_station = (
    value_sanity_flags
    .query('station == station')
    .groupby('station')['severity']
    .apply(lambda s: bool((s == 'severe').any()))
    .to_dict()
)

status_rows = []

for _, row in station_coverage_summary.iterrows():
    station_name = row['station']
    missing_rate = float(row['missing_rate'])
    pm25_missing_rate = float(pm25_missing_by_station.get(station_name, np.nan))
    duplicate_key_count = int(duplicate_key_count_by_station.get(station_name, 0))
    severe_sanity_flag = bool(severe_sanity_by_station.get(station_name, False))

    reasons = []

    if missing_rate > TH_INCLUDE_MAX_MISSING_HOUR_RATE:
        reasons.append('missing_station_hours_rate_gt_0.01')
    if missing_rate > TH_EXCLUDE_MISSING_HOUR_RATE:
        reasons.append('missing_station_hours_rate_gt_0.10')

    if np.isfinite(pm25_missing_rate) and pm25_missing_rate > TH_INCLUDE_MAX_PM25_MISSING_RATE:
        reasons.append('pm25_missing_rate_gt_0.10')
    if np.isfinite(pm25_missing_rate) and pm25_missing_rate > TH_EXCLUDE_PM25_MISSING_RATE:
        reasons.append('pm25_missing_rate_gt_0.50')

    if not np.isfinite(pm25_missing_rate):
        reasons.append('pm25_missing_rate_unavailable')

    if duplicate_key_count > 0:
        reasons.append('duplicate_keys_gt_0')

    if severe_sanity_flag:
        reasons.append('severe_value_sanity_flag')

    exclude_candidate = (
        (missing_rate > TH_EXCLUDE_MISSING_HOUR_RATE)
        or (np.isfinite(pm25_missing_rate) and pm25_missing_rate > TH_EXCLUDE_PM25_MISSING_RATE)
        or (duplicate_key_count > 0)
        or severe_sanity_flag
    )

    include_candidate = (
        (missing_rate <= TH_INCLUDE_MAX_MISSING_HOUR_RATE)
        and (np.isfinite(pm25_missing_rate) and pm25_missing_rate <= TH_INCLUDE_MAX_PM25_MISSING_RATE)
        and (duplicate_key_count == 0)
        and (not severe_sanity_flag)
    )

    if exclude_candidate:
        status = 'exclude_candidate'
    elif include_candidate:
        status = 'include'
    else:
        status = 'caution'

    status_rows.append({
        'station': station_name,
        'status': status,
        'reasons': json.dumps(reasons, ensure_ascii=False),
        'missing_rate': missing_rate,
        'pm25_missing_rate': pm25_missing_rate,
        'duplicate_key_count': duplicate_key_count,
        'severe_sanity_flag': severe_sanity_flag,
    })

station_status_policy = pd.DataFrame(status_rows).sort_values(['status', 'station']).reset_index(drop=True)

if station_status_policy.empty:
    raise ValueError('No stations available for status assignment')

if int((station_status_policy['status'] != 'exclude_candidate').sum()) == 0:
    raise ValueError('No valid stations after audit policy (all are exclude_candidate)')

station_status_policy


## 11. Persist temporal audit artifacts


In [ ]:
AUDIT_ROOT.mkdir(parents=True, exist_ok=True)

path_temporal_audit_summary = AUDIT_ROOT / 'temporal_audit_summary.json'
path_station_coverage = AUDIT_ROOT / 'station_coverage_summary.parquet'
path_station_status = AUDIT_ROOT / 'station_status_policy.parquet'
path_missing_station_hours = AUDIT_ROOT / 'missing_station_hours.parquet'
path_gap_segments = AUDIT_ROOT / 'gap_segments_top.parquet'
path_missingness_overall = AUDIT_ROOT / 'missingness_overall.parquet'
path_missingness_by_station = AUDIT_ROOT / 'missingness_by_station.parquet'
path_value_sanity_flags = AUDIT_ROOT / 'value_sanity_flags.parquet'

artifact_paths = {
    'temporal_audit_summary': path_temporal_audit_summary.relative_to(PROJECT_ROOT).as_posix(),
    'station_coverage_summary': path_station_coverage.relative_to(PROJECT_ROOT).as_posix(),
    'station_status_policy': path_station_status.relative_to(PROJECT_ROOT).as_posix(),
    'missing_station_hours': path_missing_station_hours.relative_to(PROJECT_ROOT).as_posix(),
    'gap_segments_top': path_gap_segments.relative_to(PROJECT_ROOT).as_posix(),
    'missingness_overall': path_missingness_overall.relative_to(PROJECT_ROOT).as_posix(),
    'missingness_by_station': path_missingness_by_station.relative_to(PROJECT_ROOT).as_posix(),
    'value_sanity_flags': path_value_sanity_flags.relative_to(PROJECT_ROOT).as_posix(),
}

artifact_paths


In [ ]:
station_coverage_summary.to_parquet(path_station_coverage, index=False)
station_status_policy.to_parquet(path_station_status, index=False)
gap_segments_top.to_parquet(path_gap_segments, index=False)
missingness_overall.to_parquet(path_missingness_overall, index=False)
missingness_by_station.to_parquet(path_missingness_by_station, index=False)
value_sanity_flags.to_parquet(path_value_sanity_flags, index=False)

wrote_missing_station_hours = False
if not missing_station_hours.empty:
    missing_station_hours.to_parquet(path_missing_station_hours, index=False)
    wrote_missing_station_hours = True

temporal_audit_summary = {
    'inputs': {
        'assembled_parquet': ASSEMBLED_PATH.relative_to(PROJECT_ROOT).as_posix(),
        'assembly_manifest': MANIFEST_PATH.relative_to(PROJECT_ROOT).as_posix(),
        'manifest_points_to': manifest_points_to,
        'manifest_parquet_path_match': bool(manifest_parquet_path_match),
    },
    'run_timestamp': datetime.now(timezone.utc).isoformat(),
    'thresholds': {
        'include_max_missing_hour_rate': TH_INCLUDE_MAX_MISSING_HOUR_RATE,
        'include_max_pm25_missing_rate': TH_INCLUDE_MAX_PM25_MISSING_RATE,
        'exclude_missing_hour_rate': TH_EXCLUDE_MISSING_HOUR_RATE,
        'exclude_pm25_missing_rate': TH_EXCLUDE_PM25_MISSING_RATE,
        'severe_negative_rate': TH_SEVERE_NEGATIVE_RATE,
        'severe_plausibility_rate': TH_SEVERE_PLAUSIBILITY_RATE,
        'temp_bounds_c': list(TEMP_BOUNDS_C),
        'pres_bounds_hpa': list(PRES_BOUNDS_HPA),
        'top_gaps_per_station': TOP_GAPS_PER_STATION,
    },
    'headline': {
        'n_rows': int(len(df)),
        'n_stations': int(df['station'].nunique()),
        'duplicate_key_count_overall': int(duplicate_key_count_overall),
        'total_missing_station_hours': int(len(missing_station_hours)),
        'stations_by_status': station_status_policy.groupby('status').size().astype(int).to_dict(),
        'wrote_missing_station_hours': bool(wrote_missing_station_hours),
    },
    'artifacts': artifact_paths,
}

with path_temporal_audit_summary.open('w', encoding='utf-8') as f:
    json.dump(temporal_audit_summary, f, indent=2, ensure_ascii=False)

temporal_audit_summary['headline']


## 12. Notebook close

This notebook ends after producing the temporal and structural audit artifacts.

Notebook 02 uses the persisted outputs in `data/interim/beijing_air_quality/temporal_audit/` for environmental EDA.
